# Training & Experimentation Notebook

## Google Smartphone Decimeter Challenge 2022

This notebook covers:
1. Training data generation
2. Feature engineering
3. LightGBM training with cross-validation
4. 5-seed ensemble training
5. Parameter sweeps (Mahalanobis sigma, Kalman parameters)
6. Hyperparameter tuning (2 rounds)
7. Comprehensive results and findings

All heavy computation (GNSS solving, feature extraction) is handled by `src/` modules. This notebook orchestrates training, experiments, and documents results.

### Attribution & References

The core GNSS solver is adapted from **Taro Suzuki's** (Chiba Institute of Technology) public Kaggle notebook:

- **Notebook:** [Carrier Smoothing + Robust WLS + Kalman Smoother](https://www.kaggle.com/code/taroz1461/carrier-smoothing-robust-wls-kalman-smoother)
- **Paper:** Suzuki, T. (2023). "Precise Position Estimation Using Smartphone Raw GNSS Data Based on Two-Step Optimization." *Sensors*, 23(3), 1205.

**Adapted from Suzuki:** Hatch filter carrier smoothing, robust WLS solver (scipy least_squares with Sagnac correction), outlier detection thresholds, forward-backward Kalman smoother (RTS).

**Our modifications:** Cauchy loss (replacing soft_l1), adaptive Kalman Q/R scaling, Mahalanobis gating (sigma=17), Numba JIT for Hatch filter.

**Phone-specific bias modelling** was inspired by **J.B.O. Mitchell's** ([jbomitchell](https://www.kaggle.com/jbomitchell)) notebooks:
- [Phone-Specific Bias Corrections (Clipped)](https://www.kaggle.com/code/jbomitchell/phone-specific-bias-corrections-clipped) — device-dependent lever arm and heading-based bias corrections
- [Tectonic Correction](https://www.kaggle.com/code/jbomitchell/tectonic-correction) — base station tectonic drift correction

Which built on **saitodevel01's** lever arm bias analysis from the 2021 competition:
- [GSDC - Bias EDA](https://www.kaggle.com/code/saitodevel01/gsdc-bias-eda)
- [GSDC - Bias Correction](https://www.kaggle.com/code/saitodevel01/gsdc-bias-correction)

**Additional references:** [Alejandro Cuevas (pollicio)](https://www.kaggle.com/pollicio) — competition notebooks that informed the overall approach.

**ML correction** (per-epoch LightGBM error prediction) was used by multiple top-10 finishers. This implementation is original.

In [ ]:
import sys, pathlib, time
import numpy as np, pandas as pd, joblib
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from tqdm.notebook import tqdm
from scipy.interpolate import interp1d

from src.data_loader import load_gnss_raw
from src.gnss_solver import solve_trip_robust
from src.post_processing import median_filter_trajectory, savgol_smooth_trajectory
from src.ml_correction import (
    FEATURE_COLS, PHONE_MODELS, build_training_dataset,
    add_rolling_features, haversine_m
)

ROOT = pathlib.Path("C:/Users/Max/PycharmProjects/sc400AGN")
MODEL_DIR = ROOT / "models"
DATASET_ROOT = ROOT / "kaggle_dataset"

## 1. Training Data Generation

Training data is generated by running all 170 training trips through the Hatch+Cauchy WLS pipeline with `sigma_mahalanobis=30.0`. The intentionally higher sigma (vs inference sigma=17) gives more error variation for the ML model to learn from.

- **Output:** `ml_training_data.csv` (~295K epochs, 170 trips)
- **Runtime:** ~60 minutes
- **Skip if:** CSV already exists and is up-to-date

**Key insight:** Matched sigma training (σ=17 for both training and inference) scored 2.562m — worse than mismatched (σ=30 train, σ=17 inference) at 2.525m. The extra noise in training data helps LightGBM learn larger correction patterns.

In [ ]:
# Regenerate training data (skip if CSV exists)
csv_path = ROOT / "ml_training_data.csv"
if csv_path.exists():
    print(f"Training data exists: {csv_path} ({csv_path.stat().st_size / 1e6:.1f} MB)")
    print("Delete the file and re-run this cell to regenerate.")
else:
    print("Generating training data (this takes ~60 minutes)...")
    df = build_training_dataset(DATASET_ROOT)  # default sigma=30
    df.to_csv(csv_path, index=False)
    print(f"Saved: {csv_path} ({len(df)} rows)")

## 2. Feature Engineering

### Base Features (36)
From `src/ml_correction.FEATURE_COLS`:
- Satellite counts, CN0 statistics, pseudorange uncertainties
- HDOP, residual norms, speed, elevation statistics
- Rolling features (speed_std_5, lat/lon_std_5, etc.)

### Extended Features (+10 = 46 total)
Computed during training data generation:
- `adr_frac` — fraction of satellites with valid ADR
- `n_constellations` — number of distinct constellation types
- `cn0_range` — signal quality spread (max - min CN0)
- `position_jump_m` — haversine distance from previous epoch
- `lat_accel`, `lon_accel` — second derivative of position
- `epoch_consistency_5` — position std in 5-epoch window
- `hdop_change` — epoch-to-epoch HDOP change
- `residual_mean_5` — rolling mean of WLS residual
- `pr_unc_change` — epoch-to-epoch pseudorange uncertainty change

### Round 2 Features (+8 = 54 total) — ELIMINATED
Additional features tested but hurt performance (2.537m vs 2.525m with 46 features):
- `max_elev`, `elev_range`, `cn0_q25`, `pr_unc_median`, `mean_prr_unc`, `wls_converged`, `speed_change`, `n_sats_change`

**Result:** 46 features is optimal. With only 170 training trips, additional features cause overfitting.

In [ ]:
# Define feature columns
EXTENDED_FEATURE_COLS = FEATURE_COLS + [
    # Satellite signal quality
    "adr_frac", "n_constellations", "cn0_range",
    # Trajectory dynamics
    "position_jump_m", "lat_accel", "lon_accel", "epoch_consistency_5",
    # Signal dynamics
    "hdop_change", "residual_mean_5", "pr_unc_change",
    # Round 2 features (kept for experimentation, but 46-feature model is better)
    "max_elev", "elev_range", "cn0_q25", "pr_unc_median",
    "mean_prr_unc", "wls_converged", "speed_change", "n_sats_change",
]
print(f"Base: {len(FEATURE_COLS)} | Extended: {len(FEATURE_COLS) + 10} | Full: {len(EXTENDED_FEATURE_COLS)}")

In [ ]:
# Load training data
print("Loading training data...")
df = pd.read_csv(ROOT / "ml_training_data.csv")
df["device"] = df["trip_id"].str.split("/").str[-1]
mask = df["lat_error"].notna() & df["lon_error"].notna()
df = df[mask].copy().reset_index(drop=True)

# Only keep columns that exist in CSV
cols_orig = [c for c in FEATURE_COLS if c in df.columns]
cols_ext  = [c for c in EXTENDED_FEATURE_COLS if c in df.columns]
cols_46   = [c for c in (FEATURE_COLS + EXTENDED_FEATURE_COLS[:10]) if c in df.columns]
missing = [c for c in EXTENDED_FEATURE_COLS if c not in df.columns]

print(f"  {len(df)} epochs, {df['trip_id'].nunique()} trips")
print(f"  Original features: {len(cols_orig)}/{len(FEATURE_COLS)}")
print(f"  46-feature set: {len(cols_46)}")
print(f"  Full extended: {len(cols_ext)}/{len(EXTENDED_FEATURE_COLS)}")
if missing:
    print(f"  Missing: {missing}")

X_orig = df[cols_orig].fillna(0.0)
X_46   = df[cols_46].fillna(0.0)
X_ext  = df[cols_ext].fillna(0.0)
y_lat  = df["lat_error"].values
y_lon  = df["lon_error"].values
groups = df["trip_id"].values

## 3. LightGBM Cross-Validation

5-fold GroupKFold CV comparing original (36) vs extended (46) feature sets.

**Competition metric:** mean of (p50 + p95) / 2 per trip, averaged across all trips.

Note: CV metric does not always correlate with public leaderboard. For example, matched σ=17 training improved CV but hurt public score by 0.037m.

In [ ]:
lgbm_params = dict(
    n_estimators=600, learning_rate=0.04, num_leaves=31,
    min_child_samples=40, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0, n_jobs=-1, random_state=42, verbose=-1,
)

gkf = GroupKFold(n_splits=5)
oof_lat_o = np.zeros(len(df))
oof_lon_o = np.zeros(len(df))
oof_lat_e = np.zeros(len(df))
oof_lon_e = np.zeros(len(df))

cv_lat_iters_e, cv_lon_iters_e = [], []

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_ext, y_lat, groups)):
    X_tr_o, X_val_o = X_orig.iloc[tr_idx], X_orig.iloc[val_idx]
    X_tr_e, X_val_e = X_46.iloc[tr_idx], X_46.iloc[val_idx]

    # Original feature model
    lat_o = lgb.LGBMRegressor(**lgbm_params)
    lat_o.fit(X_tr_o, y_lat[tr_idx],
              eval_set=[(X_val_o, y_lat[val_idx])],
              callbacks=[lgb.early_stopping(50, verbose=False)])
    lon_o = lgb.LGBMRegressor(**lgbm_params)
    lon_o.fit(X_tr_o, y_lon[tr_idx],
              eval_set=[(X_val_o, y_lon[val_idx])],
              callbacks=[lgb.early_stopping(50, verbose=False)])

    # Extended feature model (46 features)
    lat_e = lgb.LGBMRegressor(**lgbm_params)
    lat_e.fit(X_tr_e, y_lat[tr_idx],
              eval_set=[(X_val_e, y_lat[val_idx])],
              callbacks=[lgb.early_stopping(50, verbose=False)])
    lon_e = lgb.LGBMRegressor(**lgbm_params)
    lon_e.fit(X_tr_e, y_lon[tr_idx],
              eval_set=[(X_val_e, y_lon[val_idx])],
              callbacks=[lgb.early_stopping(50, verbose=False)])

    oof_lat_o[val_idx] = lat_o.predict(X_val_o)
    oof_lon_o[val_idx] = lon_o.predict(X_val_o)
    oof_lat_e[val_idx] = lat_e.predict(X_val_e)
    oof_lon_e[val_idx] = lon_e.predict(X_val_e)

    cv_lat_iters_e.append(lat_e.best_iteration_)
    cv_lon_iters_e.append(lon_e.best_iteration_)
    print(f"  Fold {fold}: orig lat={lat_o.best_iteration_} ext lat={lat_e.best_iteration_}")

In [ ]:
# Per-device competition metric
print("Per-device competition metric (CV):")
print(f"{'Device':<28} {'orig':>8} {'extended':>10} {'delta':>8}")
print("-" * 60)

total_orig_scores, total_ext_scores = [], []
test_dist = {
    "GooglePixel5": 0.36, "XiaomiMi8": 0.31, "SamsungGalaxyS20Ultra": 0.22,
    "GooglePixel4": 0.08, "GooglePixel6Pro": 0.03, "GooglePixel4XL": 0.00,
}
test_weighted_o, test_weighted_e = 0.0, 0.0

for dev, grp in df.groupby("device"):
    idx = grp.index.values
    gt_lat_d = grp["lat"].values - y_lat[idx]
    gt_lon_d = grp["lon"].values - y_lon[idx]

    corr_err_o = haversine_m(grp["lat"].values - oof_lat_o[idx],
                              grp["lon"].values - oof_lon_o[idx], gt_lat_d, gt_lon_d)
    corr_err_e = haversine_m(grp["lat"].values - oof_lat_e[idx],
                              grp["lon"].values - oof_lon_e[idx], gt_lat_d, gt_lon_d)

    per_trip_o, per_trip_e = [], []
    for tid, tg in grp.assign(co=corr_err_o, ce=corr_err_e).groupby("trip_id"):
        o = (np.percentile(tg.co, 50) + np.percentile(tg.co, 95)) / 2
        e = (np.percentile(tg.ce, 50) + np.percentile(tg.ce, 95)) / 2
        per_trip_o.append(o)
        per_trip_e.append(e)
        total_orig_scores.append(o)
        total_ext_scores.append(e)

    mo = np.mean(per_trip_o)
    me = np.mean(per_trip_e)
    tw = test_dist.get(dev, 0.0)
    test_weighted_o += tw * mo
    test_weighted_e += tw * me
    print(f"  {dev:<26} {mo:>8.3f} {me:>10.3f} {me-mo:>+8.3f}")

print("-" * 60)
print(f"  {'OVERALL':<26} {np.mean(total_orig_scores):>8.3f} {np.mean(total_ext_scores):>10.3f} "
      f"{np.mean(total_ext_scores)-np.mean(total_orig_scores):>+8.3f}")
print(f"\nTest-weighted:  orig={test_weighted_o:.3f}m  ext={test_weighted_e:.3f}m  "
      f"delta={test_weighted_e-test_weighted_o:+.3f}m")

In [ ]:
# Feature importances (extended model, last fold)
print("Top-20 feature importances:")
imp = pd.Series(lat_e.feature_importances_, index=cols_46[:len(lat_e.feature_importances_)]).sort_values(ascending=False)
for feat, val in imp.head(20).items():
    marker = " *** NEW ***" if feat not in FEATURE_COLS else ""
    print(f"  {feat:<30} {val:>8.0f}{marker}")

## 4. Train Final Single Models

Use median of CV fold iterations for stable n_estimators (early stopping on small holdouts is noisy).

In [ ]:
best_lat_n = max(50, int(np.median(cv_lat_iters_e)))
best_lon_n = max(50, int(np.median(cv_lon_iters_e)))
print(f"CV lat iters: {cv_lat_iters_e} -> n={best_lat_n}")
print(f"CV lon iters: {cv_lon_iters_e} -> n={best_lon_n}")

final_lat_params = dict(**lgbm_params, n_estimators=best_lat_n)
final_lon_params = dict(**lgbm_params, n_estimators=best_lon_n)

lat_final = lgb.LGBMRegressor(**final_lat_params)
lat_final.fit(X_46, y_lat)

lon_final = lgb.LGBMRegressor(**final_lon_params)
lon_final.fit(X_46, y_lon)

print(f"Final: lat_n={best_lat_n}, lon_n={best_lon_n}")

# Save
joblib.dump(lat_final, MODEL_DIR / "lat_model_ext.joblib")
joblib.dump(lon_final, MODEL_DIR / "lon_model_ext.joblib")
joblib.dump(cols_46, MODEL_DIR / "feature_cols_ext.joblib")
print(f"Saved to models/: lat_model_ext, lon_model_ext, feature_cols_ext ({len(cols_46)} features)")

## 5. 5-Seed Ensemble Training

Averaging predictions from 5 differently-seeded LightGBM models reduces variance. Each model sees the same data but makes different splitting decisions due to random subsampling.

- **Seeds:** 42, 123, 456, 789, 1024
- **Key param:** `min_child_samples=65` (from hyperparameter tuning)
- **Output:** `lat_models_ensemble.joblib`, `lon_models_ensemble.joblib`

In [ ]:
SEEDS = [42, 123, 456, 789, 1024]

ens_params = dict(
    learning_rate=0.04, num_leaves=31,
    min_child_samples=65, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0, n_jobs=-1, verbose=-1,
)

lat_models, lon_models = [], []
for seed in SEEDS:
    p = dict(ens_params, random_state=seed, n_estimators=best_lat_n)
    lat_m = lgb.LGBMRegressor(**p)
    lat_m.fit(X_46, y_lat)

    p_lon = dict(ens_params, random_state=seed, n_estimators=best_lon_n)
    lon_m = lgb.LGBMRegressor(**p_lon)
    lon_m.fit(X_46, y_lon)

    lat_models.append(lat_m)
    lon_models.append(lon_m)
    print(f"  Seed {seed}: lat={lat_m.n_estimators_}, lon={lon_m.n_estimators_}")

joblib.dump(lat_models, MODEL_DIR / "lat_models_ensemble.joblib")
joblib.dump(lon_models, MODEL_DIR / "lon_models_ensemble.joblib")
print(f"\nSaved {len(lat_models)}-seed ensemble to models/")

## 6. Reusable Submission Helper

The sigma sweep and Kalman sweep share 90% of their trip-processing loop. This helper function generates a submission DataFrame for any parameter combination.

In [ ]:
def generate_submission_for_params(
    sigma=17.0, speed_q_ref=5.0, hdop_r_ref=1.5,
    lat_ens=None, lon_ens=None,
    feature_cols=None, sample_ts=None, test_dir=None,
):
    """Process all test trips and return submission DataFrame."""
    all_rows = []
    n_total = 0

    for trip_dir in tqdm(sorted(test_dir.iterdir()), desc=f"s={sigma} q={speed_q_ref} h={hdop_r_ref}"):
        if not trip_dir.is_dir():
            continue
        for dev_dir in sorted(trip_dir.iterdir()):
            if not dev_dir.is_dir():
                continue
            trip_id = f"{trip_dir.name}/{dev_dir.name}"
            device_name = dev_dir.name

            try:
                gnss_df = load_gnss_raw(dev_dir)
                if len(gnss_df) == 0:
                    continue

                result, feat_df = solve_trip_robust(
                    gnss_df, device_name=device_name,
                    collect_features=True, use_tdcp=False,
                    sigma_mahalanobis=sigma,
                    speed_q_ref=speed_q_ref, hdop_r_ref=hdop_r_ref)

                if len(result) == 0 or feat_df is None or len(feat_df) == 0:
                    continue

                pos_df = pd.DataFrame({
                    "UnixTimeMillis": result["epoch_ms"].values.astype(np.int64),
                    "LatitudeDegrees": result["lat"].values,
                    "LongitudeDegrees": result["lon"].values,
                })
                pos_df["LatitudeDegrees"] = pos_df["LatitudeDegrees"].interpolate().ffill().bfill()
                pos_df["LongitudeDegrees"] = pos_df["LongitudeDegrees"].interpolate().ffill().bfill()
                pos_df = median_filter_trajectory(pos_df, kernel_size=3)

                feat_df["lat"] = pos_df["LatitudeDegrees"].values[:len(feat_df)]
                feat_df["lon"] = pos_df["LongitudeDegrees"].values[:len(feat_df)]
                feat_df["phone_model"] = PHONE_MODELS.get(device_name, -1)
                feat_df = add_rolling_features(feat_df)

                cols_present = [c for c in feature_cols if c in feat_df.columns]
                X = feat_df[cols_present].fillna(0.0)
                pred_lat_err = np.mean([m.predict(X) for m in lat_ens], axis=0)
                pred_lon_err = np.mean([m.predict(X) for m in lon_ens], axis=0)

                n = min(len(pred_lat_err), len(pos_df))
                pos_df = pos_df.iloc[:n].copy()
                pos_df["LatitudeDegrees"] = pos_df["LatitudeDegrees"].values - pred_lat_err[:n]
                pos_df["LongitudeDegrees"] = pos_df["LongitudeDegrees"].values - pred_lon_err[:n]

                pos_df = savgol_smooth_trajectory(pos_df, window_length=7, polyorder=2)

                if trip_id in sample_ts:
                    target_ts = sample_ts[trip_id]
                    pred_t = pos_df["UnixTimeMillis"].values
                    pred_lat = pos_df["LatitudeDegrees"].values
                    pred_lon = pos_df["LongitudeDegrees"].values

                    if len(pred_t) >= 2:
                        f_lat = interp1d(pred_t, pred_lat, kind="linear",
                                         bounds_error=False, fill_value=(pred_lat[0], pred_lat[-1]))
                        f_lon = interp1d(pred_t, pred_lon, kind="linear",
                                         bounds_error=False, fill_value=(pred_lon[0], pred_lon[-1]))
                        out_lat = f_lat(target_ts)
                        out_lon = f_lon(target_ts)
                    else:
                        out_lat = np.full(len(target_ts), pred_lat[0] if len(pred_lat) else 0.0)
                        out_lon = np.full(len(target_ts), pred_lon[0] if len(pred_lon) else 0.0)

                    for ts, la, lo in zip(target_ts, out_lat, out_lon):
                        all_rows.append({
                            "tripId": trip_id,
                            "UnixTimeMillis": int(ts),
                            "LatitudeDegrees": la,
                            "LongitudeDegrees": lo,
                        })
                n_total += 1

            except Exception as exc:
                import traceback
                print(f"\nError {trip_id}: {exc}")
                traceback.print_exc()

    return pd.DataFrame(all_rows), n_total


# Load models and sample timestamps for sweeps
print("Loading models for parameter sweeps...")
lat_models_ens = joblib.load(MODEL_DIR / "lat_models_ensemble.joblib")
lon_models_ens = joblib.load(MODEL_DIR / "lon_models_ensemble.joblib")
feature_cols_ext = joblib.load(MODEL_DIR / "feature_cols_ext.joblib")
print(f"  Ensemble: {len(lat_models_ens)} models, {len(feature_cols_ext)} features")

sample_df = pd.read_csv(DATASET_ROOT / "sample_submission.csv")
sample_ts = {}
for trip_id, grp in sample_df.groupby("tripId"):
    sample_ts[trip_id] = np.sort(grp["UnixTimeMillis"].values)

test_dir = DATASET_ROOT / "test"

## 7. Mahalanobis Sigma Sweep

The Mahalanobis distance threshold (`sigma_mahalanobis`) controls outlier rejection in the Kalman filter. Lower values reject more measurements aggressively; higher values are more permissive.

**Tested range:** 3.0 to 30.0
**Optimal:** sigma=17.0 (public score 2.525m)

Key findings:
- sigma < 10: too aggressive, rejects valid measurements
- sigma = 17: sweet spot for public leaderboard
- sigma > 20: too permissive, admits outlier epochs
- sigma = 30: default, used for training data (more noise = better ML training)

In [ ]:
SIGMAS = [4, 6, 7, 8, 9, 10, 12, 15, 16, 17, 18, 20, 25, 30]

for sigma in SIGMAS:
    print(f"\n=== sigma={sigma} ===")
    out_df, n_total = generate_submission_for_params(
        sigma=sigma,
        lat_ens=lat_models_ens, lon_ens=lon_models_ens,
        feature_cols=feature_cols_ext, sample_ts=sample_ts, test_dir=test_dir,
    )
    out_path = ROOT / f"submission_sigma{sigma:.0f}.csv"
    out_df.to_csv(out_path, index=False)
    print(f"  Saved: {out_path.name} ({len(out_df)} rows, {n_total} trips, {out_df.isnull().sum().sum()} NaN)")

### Sigma Sweep Results

| Sigma | Public | Private | Notes |
|---|---|---|---|
| 4.0 | 2.615 | 2.429 | |
| 6.0 | 2.551 | 2.452 | |
| 7.0 | 2.552 | 2.430 | |
| 8.0 | 2.559 | 2.437 | |
| 9.0 | 2.549 | 2.440 | |
| 10.0 | 2.540 | 2.450 | |
| 12.0 | 2.535 | 2.489 | |
| 15.0 | 2.532 | 2.472 | |
| 16.0 | 2.527 | 2.476 | |
| **17.0** | **2.525** | **2.474** | **Optimal** |
| 18.0 | 2.529 | 2.464 | |
| 20.0 | 2.579 | 2.471 | |
| 25.0 | 2.621 | - | Previous best before sweep |
| 30.0 | ~2.65 | - | Default (used for training) |

**Observation:** Public and private scores do not perfectly correlate. sigma=7 has best private (2.430) but sigma=17 has best public (2.525). Since we optimize for public score, sigma=17 is used.

## 8. Kalman Parameter Sweep

Two Kalman smoother parameters control process noise:
- `speed_q_ref` — reference speed for Q-matrix scaling. Higher speed = more process noise.
- `hdop_r_ref` — reference HDOP for R-matrix scaling. Higher HDOP = more measurement noise.

**Baseline:** speed_q_ref=5.0, hdop_r_ref=1.5 (with sigma=17)
**Result:** Defaults are optimal. All variations hurt performance.

In [ ]:
CONFIGS = [
    ("sqr2",   17.0, 2.0, 1.5),
    ("sqr3",   17.0, 3.0, 1.5),
    ("sqr10",  17.0, 10.0, 1.5),
    ("sqr20",  17.0, 20.0, 1.5),
    ("hdop1",  17.0, 5.0, 1.0),
    ("hdop2",  17.0, 5.0, 2.0),
    ("hdop3",  17.0, 5.0, 3.0),
    ("hdop5",  17.0, 5.0, 5.0),
    ("noQ",    17.0, 999.0, 1.5),    # disable speed scaling
    ("noHDOP", 17.0, 5.0, 999.0),   # disable HDOP scaling
]

for name, sigma, sqr, hdr in CONFIGS:
    print(f"\n=== {name}: sigma={sigma}, speed_q_ref={sqr}, hdop_r_ref={hdr} ===")
    out_df, n_total = generate_submission_for_params(
        sigma=sigma, speed_q_ref=sqr, hdop_r_ref=hdr,
        lat_ens=lat_models_ens, lon_ens=lon_models_ens,
        feature_cols=feature_cols_ext, sample_ts=sample_ts, test_dir=test_dir,
    )
    out_path = ROOT / f"submission_kalman_{name}.csv"
    out_df.to_csv(out_path, index=False)
    print(f"  Saved: {out_path.name} ({len(out_df)} rows, {n_total} trips)")

### Kalman Parameter Results

| Config | speed_q_ref | hdop_r_ref | Public | Private |
|---|---|---|---|---|
| sqr2 | 2.0 | 1.5 | 2.588 | 2.464 |
| sqr3 | 3.0 | 1.5 | 2.571 | 2.461 |
| **baseline** | **5.0** | **1.5** | **2.525** | **2.474** |
| sqr10 | 10.0 | 1.5 | 2.541 | 2.476 |
| sqr20 | 20.0 | 1.5 | 2.580 | 2.465 |
| noQ | 999 | 1.5 | 2.604 | 2.466 |
| hdop1 | 5.0 | 1.0 | 2.601 | 2.474 |
| hdop2 | 5.0 | 2.0 | 2.537 | 2.473 |
| hdop3 | 5.0 | 3.0 | 2.559 | 2.484 |
| hdop5 | 5.0 | 5.0 | 2.578 | 2.532 |
| noHDOP | 5.0 | 999 | 2.632 | 2.577 |

**Conclusion:** Default parameters (5.0, 1.5) are optimal. All changes hurt public score.

## 9. Hyperparameter Tuning — Round 1

One-at-a-time sweep of 7 LightGBM parameters. Each parameter is varied independently while others stay at baseline.

**Baseline params:** n_estimators=600, lr=0.04, num_leaves=31, min_child_samples=40, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0

In [ ]:
def cv_score(params, n_splits=5):
    """Run GroupKFold CV and return competition metric."""
    gkf = GroupKFold(n_splits=n_splits)
    oof_lat = np.zeros(len(df))
    oof_lon = np.zeros(len(df))
    lat_iters, lon_iters = [], []

    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_46, y_lat, groups)):
        X_tr, X_val = X_46.iloc[tr_idx], X_46.iloc[val_idx]

        lat_m = lgb.LGBMRegressor(**params)
        lat_m.fit(X_tr, y_lat[tr_idx],
                  eval_set=[(X_val, y_lat[val_idx])],
                  callbacks=[lgb.early_stopping(50, verbose=False)])

        lon_m = lgb.LGBMRegressor(**params)
        lon_m.fit(X_tr, y_lon[tr_idx],
                  eval_set=[(X_val, y_lon[val_idx])],
                  callbacks=[lgb.early_stopping(50, verbose=False)])

        oof_lat[val_idx] = lat_m.predict(X_val)
        oof_lon[val_idx] = lon_m.predict(X_val)
        lat_iters.append(lat_m.best_iteration_)
        lon_iters.append(lon_m.best_iteration_)

    # Competition metric
    corr_lat = df["lat"].values - oof_lat
    corr_lon = df["lon"].values - oof_lon
    gt_lat = df["lat"].values - y_lat
    gt_lon = df["lon"].values - y_lon
    err = haversine_m(corr_lat, corr_lon, gt_lat, gt_lon)

    trip_scores = []
    for tid in df["trip_id"].unique():
        mask_t = df["trip_id"].values == tid
        e = err[mask_t]
        trip_scores.append((np.percentile(e, 50) + np.percentile(e, 95)) / 2)

    return np.mean(trip_scores), lat_iters, lon_iters

In [ ]:
baseline_params = dict(
    n_estimators=600, learning_rate=0.04, num_leaves=31,
    min_child_samples=40, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0, n_jobs=-1, random_state=42, verbose=-1,
)

print("=== Baseline ===")
t0 = time.time()
base_score, base_lat_it, base_lon_it = cv_score(baseline_params)
print(f"  Score: {base_score:.4f}m  ({time.time()-t0:.0f}s)")

# One-at-a-time sweep
param_grid = {
    "num_leaves":       [15, 31, 63, 127],
    "min_child_samples": [20, 40, 80, 160],
    "learning_rate":    [0.01, 0.02, 0.04, 0.08],
    "subsample":        [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_alpha":        [0.0, 0.1, 1.0],
    "reg_lambda":       [0.1, 1.0, 10.0],
}

results = [("baseline", baseline_params, base_score, base_lat_it, base_lon_it)]

print("\n=== One-at-a-time parameter sweep ===")
for param_name, values in param_grid.items():
    print(f"\n--- {param_name} ---")
    for val in values:
        p = dict(baseline_params)
        p[param_name] = val
        label = f"{param_name}={val}"

        if p == baseline_params:
            print(f"  {label:40s} -> {base_score:.4f}m (baseline)")
            continue

        t0 = time.time()
        score, lat_it, lon_it = cv_score(p)
        delta = score - base_score
        marker = " *** BEST ***" if delta < -0.01 else ""
        print(f"  {label:40s} -> {score:.4f}m ({delta:+.4f}){marker}  ({time.time()-t0:.0f}s)")
        results.append((label, p, score, lat_it, lon_it))

# Summary
print("\n=== RESULTS RANKED ===")
results.sort(key=lambda x: x[2])
for i, (label, p, score, lat_it, lon_it) in enumerate(results[:15]):
    delta = score - base_score
    print(f"  {i+1:2d}. {score:.4f}m ({delta:+.4f})  {label}")

## 10. Hyperparameter Tuning — Round 2

Combine best parameters from Round 1 and test all combinations. Winner: `min_child_samples=65` (fine-tuned around the mcs=80 winner from R1).

In [ ]:
combos = [
    ("baseline", dict(n_estimators=600, learning_rate=0.04, num_leaves=31,
                      min_child_samples=40, subsample=0.8, colsample_bytree=0.8,
                      reg_alpha=0.1, reg_lambda=1.0, n_jobs=-1, random_state=42, verbose=-1)),
    ("mcs80", dict(n_estimators=600, learning_rate=0.04, num_leaves=31,
                   min_child_samples=80, subsample=0.8, colsample_bytree=0.8,
                   reg_alpha=0.1, reg_lambda=1.0, n_jobs=-1, random_state=42, verbose=-1)),
    ("rl0.1", dict(n_estimators=600, learning_rate=0.04, num_leaves=31,
                   min_child_samples=40, subsample=0.8, colsample_bytree=0.8,
                   reg_alpha=0.1, reg_lambda=0.1, n_jobs=-1, random_state=42, verbose=-1)),
    ("mcs80+rl0.1", dict(n_estimators=600, learning_rate=0.04, num_leaves=31,
                         min_child_samples=80, subsample=0.8, colsample_bytree=0.8,
                         reg_alpha=0.1, reg_lambda=0.1, n_jobs=-1, random_state=42, verbose=-1)),
    ("mcs80+rl10", dict(n_estimators=600, learning_rate=0.04, num_leaves=31,
                        min_child_samples=80, subsample=0.8, colsample_bytree=0.8,
                        reg_alpha=0.1, reg_lambda=10.0, n_jobs=-1, random_state=42, verbose=-1)),
    ("mcs80+cs1.0", dict(n_estimators=600, learning_rate=0.04, num_leaves=31,
                         min_child_samples=80, subsample=0.8, colsample_bytree=1.0,
                         reg_alpha=0.1, reg_lambda=1.0, n_jobs=-1, random_state=42, verbose=-1)),
    ("mcs80+rl0.1+cs1.0", dict(n_estimators=600, learning_rate=0.04, num_leaves=31,
                                min_child_samples=80, subsample=0.8, colsample_bytree=1.0,
                                reg_alpha=0.1, reg_lambda=0.1, n_jobs=-1, random_state=42, verbose=-1)),
    ("mcs80+rl10+cs1.0", dict(n_estimators=600, learning_rate=0.04, num_leaves=31,
                               min_child_samples=80, subsample=0.8, colsample_bytree=1.0,
                               reg_alpha=0.1, reg_lambda=10.0, n_jobs=-1, random_state=42, verbose=-1)),
    ("mcs80+lr0.02", dict(n_estimators=600, learning_rate=0.02, num_leaves=31,
                          min_child_samples=80, subsample=0.8, colsample_bytree=0.8,
                          reg_alpha=0.1, reg_lambda=1.0, n_jobs=-1, random_state=42, verbose=-1)),
    ("mcs80+rl0.1+lr0.02", dict(n_estimators=600, learning_rate=0.02, num_leaves=31,
                                 min_child_samples=80, subsample=0.8, colsample_bytree=0.8,
                                 reg_alpha=0.1, reg_lambda=0.1, n_jobs=-1, random_state=42, verbose=-1)),
    ("mcs60", dict(n_estimators=600, learning_rate=0.04, num_leaves=31,
                   min_child_samples=60, subsample=0.8, colsample_bytree=0.8,
                   reg_alpha=0.1, reg_lambda=1.0, n_jobs=-1, random_state=42, verbose=-1)),
    ("mcs100", dict(n_estimators=600, learning_rate=0.04, num_leaves=31,
                    min_child_samples=100, subsample=0.8, colsample_bytree=0.8,
                    reg_alpha=0.1, reg_lambda=1.0, n_jobs=-1, random_state=42, verbose=-1)),
    ("mcs120", dict(n_estimators=600, learning_rate=0.04, num_leaves=31,
                    min_child_samples=120, subsample=0.8, colsample_bytree=0.8,
                    reg_alpha=0.1, reg_lambda=1.0, n_jobs=-1, random_state=42, verbose=-1)),
    ("mcs80+lr0.02+n1200", dict(n_estimators=1200, learning_rate=0.02, num_leaves=31,
                                 min_child_samples=80, subsample=0.8, colsample_bytree=0.8,
                                 reg_alpha=0.1, reg_lambda=1.0, n_jobs=-1, random_state=42, verbose=-1)),
    ("mcs80+rl0.1+lr0.01+n1200", dict(n_estimators=1200, learning_rate=0.01, num_leaves=31,
                                        min_child_samples=80, subsample=0.8, colsample_bytree=0.8,
                                        reg_alpha=0.1, reg_lambda=0.1, n_jobs=-1, random_state=42, verbose=-1)),
]

print(f"=== Round 2: {len(combos)} combinations ===")
r2_results = []
for name, params in combos:
    t0 = time.time()
    score, lat_it, lon_it = cv_score(params)
    elapsed = time.time() - t0
    r2_results.append((name, params, score, lat_it, lon_it))
    print(f"  {name:40s} -> {score:.4f}m  ({elapsed:.0f}s)")

# Rank
r2_results.sort(key=lambda x: x[2])
r2_base = [r for r in r2_results if r[0] == "baseline"][0][2]

print(f"\n=== RANKED (baseline={r2_base:.4f}m) ===")
for i, (name, p, score, lat_it, lon_it) in enumerate(r2_results):
    delta = score - r2_base
    print(f"  {i+1:2d}. {score:.4f}m ({delta:+.4f})  {name}")

best_name, best_params_r2, best_score_r2, best_lat_it_r2, best_lon_it_r2 = r2_results[0]
print(f"\nBest: {best_name} -> {best_score_r2:.4f}m")

## 11. Comprehensive Results Summary

### What Works

- **Hatch filter carrier smoothing (N=1000):** Proper Hatch filter replaces constant-bias ADR correction. N=50->2.835, N=100->2.804, N=200->2.761, N=500->2.718, N=1000->2.717 (converged). Uses numba @njit.
- **Cauchy robust WLS loss:** Changed `loss="soft_l1"` to `loss="cauchy"` in both position AND velocity least_squares. cauchy pos only: 2.691, cauchy pos+vel: **2.466m**. f_scale=1.0 optimal.
- **Mahalanobis sigma=17:** Tested 3-30. Best public 2.525m. Sweet spot between rejecting outliers and keeping valid measurements.
- **5-seed LightGBM ensemble:** Seeds 42,123,456,789,1024. Reduces prediction variance.
- **min_child_samples=65:** From 2-round hyperparameter tuning. Prevents overfitting with only 170 training trips.
- **Mismatched training sigma (30 vs 17):** More error variation in training helps ML learn larger correction patterns.
- **46 extended features:** 36 base + 10 computed. Captures trajectory dynamics and signal quality.
- **SavGol smoothing (w=7, poly=2):** Gentle smoothing removes high-frequency noise without over-smoothing.
- **Default Kalman params:** speed_q_ref=5.0, hdop_r_ref=1.5. All alternatives tested, none better.

### What Doesn't Work

| Approach | Score | Why It Failed |
|---|---|---|
| TDCP inference | 3.4-3.8m private | Diverges on ~4 private trips; undetectable |
| L1/L5 iono-free combination | 3.850m | IonosphericDelayMeters already signal-specific; IF amplifies noise 2.6x |
| Per-satellite RAIM | 2.969m | Cauchy loss already handles outlier satellites |
| Huber loss | 2.982m | Much worse than cauchy |
| Elevation-dependent sat weighting | 2.843m | Cauchy already downweights bad sats |
| Matched sigma training (sigma=17) | 2.562m | Less error variation -> ML learns less |
| 54-feature model | 2.537m | Overfitting with 170 training trips |
| Q-scaling inversion | 3.090m | Higher speed -> more Q is correct |
| Bearing features | 2.933m | Hurt private score |
| MAE LightGBM objective | good CV | Terrible generalization to public LB |
| SavGol w=9 | 2.830m | Too much smoothing |

### Full Score History

| Submission | Public | Private | Notes |
|---|---|---|---|
| Hatch N=1000 + Cauchy pos+vel | 2.466 | - | Best private |
| Sigma=4.0 | 2.615 | 2.429 | |
| Sigma=6.0 | 2.551 | 2.452 | |
| Sigma=7.0 | 2.552 | 2.430 | |
| Sigma=8.0 | 2.559 | 2.437 | |
| Sigma=9.0 | 2.549 | 2.440 | |
| Sigma=10.0 | 2.540 | 2.450 | |
| Sigma=12.0 | 2.535 | 2.489 | |
| Sigma=15.0 | 2.532 | 2.472 | |
| Sigma=16.0 | 2.527 | 2.476 | |
| **Sigma=17.0** | **2.525** | **2.474** | **Best single public** |
| Sigma=18.0 | 2.529 | 2.464 | |
| Sigma=20.0 | 2.579 | 2.471 | |
| Sigma blend avg(12-18) | **2.524** | 2.473 | **Best public overall** |
| Sigma blend weighted | 2.524 | 2.469 | |
| Kalman sqr2 | 2.588 | 2.464 | |
| Kalman sqr3 | 2.571 | 2.461 | |
| Kalman sqr10 | 2.541 | 2.476 | |
| Kalman sqr20 | 2.580 | 2.465 | |
| Kalman noQ | 2.604 | 2.466 | |
| Kalman hdop1 | 2.601 | 2.474 | |
| Kalman hdop2 | 2.537 | 2.473 | |
| Kalman hdop3 | 2.559 | 2.484 | |
| Kalman hdop5 | 2.578 | 2.532 | |
| Kalman noHDOP | 2.632 | 2.577 | |
| Elev-weighted WLS | 2.843 | 2.643 | |
| 54-feat matched sigma | 2.562 | 2.515 | |
| 54-feat sigma=30 | 2.537 | 2.505 | |
| Mahalanobis=3.0 | 2.740 | 2.472 | |
| Q-scaling inversion | 3.090 | - | |
| Baseline (2816) | 2.816 | - | Pre-Hatch/cauchy |